In [ ]:
# SPDX-License-Identifier: Apache-2.0 AND CC-BY-NC-4.0
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Solution to the UST Exercise

In [ ]:
import cupy as cp
import cupyx.scipy.sparse as sp
from cupyx.profiler import benchmark

import matplotlib.pyplot as plt

from nvmath.sparse.generic import Matmul
from nvmath.sparse.ust import NamedFormats, Tensor

# Construct the tri-diagonal matrix and CuPy DIA/CSR/COO formats.
n = 1024 * 8
values = cp.array([[0.0] + [-1.0] * (n - 1), [4.0] * n, [1.0] * (n - 1) + [0.0]], dtype=cp.float32)
offsets = cp.array([-1, 0, 1], dtype=cp.int32)
a_cp_dia = sp.dia_matrix((values, offsets), shape=(n, n))
a_cp_csr = sp.csr_matrix(a_cp_dia)
a_cp_coo = sp.coo_matrix(a_cp_dia)
a_cp_coo.sum_duplicates()   # coalesce COO format
cp_ones = cp.ones((n,), dtype=cp.float32)
cp_out = cp.zeros((n,), dtype=cp.float32)

# Construct the UST views into those formats.
u_dia = Tensor.from_package(a_cp_dia)
u_csr = Tensor.from_package(a_cp_csr)
u_coo = Tensor.from_package(a_cp_coo)
u_ones = Tensor.from_package(cp_ones)
u_out = Tensor.from_package(cp_out)

# SOLUTION
# Convert u_coo to u_dcsr.
u_dcsr = u_coo.convert(tensor_format=NamedFormats.DCSR)

with Matmul(u_dia, u_ones, u_out) as mm:
    mm.plan()
    ust1 = benchmark(mm.execute, name="UST DIA", n_repeat=10)
    
with Matmul(u_csr, u_ones, u_out) as mm:
    mm.plan()
    ust2 = benchmark(mm.execute, name="UST CSR", n_repeat=10)    
    
with Matmul(u_coo, u_ones, u_out) as mm:
    mm.plan()
    ust3 = benchmark(mm.execute, name="UST COO", n_repeat=10)

# SOLUTION
# Create a Matmul object, plan, and benchmark.
with Matmul(u_dcsr, u_ones, u_out) as mm:
    mm.plan()
    ust4 = benchmark(mm.execute, name="UST DCSR", n_repeat=10)

print(ust1)
print(ust2)
print(ust3)   
print(ust4)   

# SOLUTION
# Plot DIA/CSR/COO/DCSR results.
labels = [ust1.name, ust2.name, ust3.name, ust4.name]
runtimes = [
    min(ust1.gpu_times[0]) * 1e6,
    min(ust2.gpu_times[0]) * 1e6,
    min(ust3.gpu_times[0]) * 1e6,
    min(ust4.gpu_times[0]) * 1e6,
]
bar_colors = ["#76B900", "#76B900", "#76B900", "#76B900"]
hatch = ["+", ".", "/", "x"]
plt.figure(figsize=(8, 6))
plt.title(f"SpMV runtime for n={n}")
plt.bar(labels, runtimes, color=bar_colors, hatch=hatch)
plt.ylabel("Execution Time (us)")
plt.xticks(rotation=45)
plt.show()

# Solution to the UST Bonus Exercise

In [ ]:
import cupy as cp

from nvmath.sparse.ust import NamedFormats, LevelFormat, TensorFormat, Tensor

strips = cp.array([ [1,  2,  3,  4,  0,  0,  0,  0,  0,  0,  0,  0], 
                    [0,  0,  0,  0,  0,  0,  0,  0,  5,  6,  7,  8], 
                    [0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
                    [0,  0,  0,  0,  9, 10, 11, 12,  0,  0,  0,  0]], dtype=cp.float32)

u_dense = Tensor.from_package(strips)


# Design a strip DSL for the UST.
i, j = NamedFormats.i, NamedFormats.j
strip_format = TensorFormat(
            [i, j],
            {
                i: LevelFormat.DENSE,
                j // 4: LevelFormat.COMPRESSED,
                j % 4: LevelFormat.DENSE,
            },
            name="strip_of_4",
        )

# Convert the array above into that format.
u_strip = u_dense.convert(tensor_format=strip_format)

# Print the result to see how many bytes are used.
print(u_strip)